In [7]:
from qdrant_manager import QdrantVectorDB


In [8]:
qdrant_vectordb = QdrantVectorDB()
client = qdrant_vectordb.client
collection_name = qdrant_vectordb.collection_name

INFO:root:✓ Kết nối đến Qdrant server tại localhost:6333 thành công
INFO:httpx:HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2 "HTTP/1.1 200 OK"
INFO:root:✓ Tạo collection 'audio_features_v2' thành công


INFO:httpx:HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


In [9]:
print(collection_name)

audio_features_v2


In [3]:
from audio_processor_v2 import AudioPreprocessor
audio_preprocessor = AudioPreprocessor()

In [10]:
from pathlib import Path
import pandas as pd

# 1. Đọc file CSV chứa nhãn và tạo dictionary map {tên_file: nhãn}
csv_path = Path(r"d:\Project\HCSDLDPT\wind_instruments_2s.csv")
df_csv = pd.read_csv(csv_path)
# Tạo mapping nhanh từ fname sang label
label_map = dict(zip(df_csv['fname'], df_csv['label']))

# Lấy tất cả file audio từ thư mục
audio_dir = Path(r"C:\Users\ADZ\Downloads\wind_instruments_2s")
all_audio_files = sorted(audio_dir.glob("*.wav"))[:500]  # Lấy 30 file đầu tiên để xử lý

# Xử lý 30 file audio
results = []
for i, audio_file in enumerate(all_audio_files, 1):
    features = audio_preprocessor.preprocess(str(audio_file))
    if features is not None:
        # Lấy nhãn từ mapping (nếu không tìm thấy sẽ trả về 'Unknown')
        file_label = label_map.get(audio_file.name, "Unknown")
        
        results.append({
            'file_name': audio_file.name,
            'file_path': str(audio_file),
            'features': features,
            'features_shape': features.shape,
            'label': file_label,  # Bổ sung key label lấy từ file CSV
        })
        print(f"[{i}/30] ✓ {audio_file.name} - Shape: {features.shape} - Label: {file_label}")
    else:
        print(f"[{i}/30] ✗ {audio_file.name} - Lỗi xử lý")

print(f"\n✓ Thành công: {len(results)}/30 file")

# Hiển thị kết quả
df_results = pd.DataFrame([
    {
        'STT': i+1,
        'File Name': r['file_name'],
        'Feature Dim': r['features_shape'][0],
        'Label': r['label']  # Hiển thị thêm cột Label
    } 
    for i, r in enumerate(results)
])

print("\n" + "="*60)
print("KẾT QUẢ XỬ LÝ 30 FILE ÂM THANH ĐẦU TIÊN VỚI NHÃN (LABEL)")
print("="*60)
print(df_results.to_string(index=False))
print(f"\nTổng cộng: {len(results)} file được xử lý thành công")
print(f"Mỗi file được biểu diễn bằng vector {results[0]['features_shape'][0]} chiều")


[1/30] ✓ 001ca53d.wav - Shape: (58,) - Label: Saxophone
[2/30] ✓ 00c9e799.wav - Shape: (58,) - Label: Oboe
[3/30] ✓ 015cf474.wav - Shape: (58,) - Label: Clarinet
[4/30] ✓ 02ac1057.wav - Shape: (58,) - Label: Flute
[5/30] ✓ 02f8e94d.wav - Shape: (58,) - Label: Clarinet
[6/30] ✓ 034e4ffa.wav - Shape: (58,) - Label: Trumpet
[7/30] ✓ 0395ba61.wav - Shape: (58,) - Label: Oboe
[8/30] ✓ 04076350.wav - Shape: (58,) - Label: Clarinet
[9/30] ✓ 04490642.wav - Shape: (58,) - Label: Trumpet
[10/30] ✓ 05a3154f.wav - Shape: (58,) - Label: Oboe
[11/30] ✓ 05d0dfa7.wav - Shape: (58,) - Label: Oboe
[12/30] ✓ 05d54a68.wav - Shape: (58,) - Label: Trumpet
[13/30] ✓ 05e4028a.wav - Shape: (58,) - Label: Saxophone
[14/30] ✓ 060e13d6.wav - Shape: (58,) - Label: Trumpet
[15/30] ✓ 061cf689.wav - Shape: (58,) - Label: Trumpet
[16/30] ✓ 064cfad3.wav - Shape: (58,) - Label: Clarinet
[17/30] ✓ 06d164df.wav - Shape: (58,) - Label: Clarinet
[18/30] ✓ 06f60188.wav - Shape: (58,) - Label: Saxophone
[19/30] ✓ 0720ef3b.wav

In [ ]:
from pathlib import Path
import pandas as pd

# Lấy tất cả file audio từ thư mục
audio_dir = Path(r"C:\Users\ADZ\Downloads\wind_instruments_2s")
all_audio_files = sorted(audio_dir.glob("*.wav"))  # Lấy 30 file đầu tiên

# Xử lý 30 file audio
results = []
for i, audio_file in enumerate(all_audio_files, 1):
    features = audio_preprocessor.preprocess(str(audio_file))
    if features is not None:
        results.append({
            'file_name': audio_file.name,
            'file_path': str(audio_file),
            'features': features,
            'features_shape': features.shape,
        })
        print(f"[{i}/30] ✓ {audio_file.name} - Shape: {features.shape}")
    else:
        print(f"[{i}/30] ✗ {audio_file.name} - Lỗi xử lý")

print(f"\n✓ Thành công: {len(results)}/30 file")

# Hiển thị kết quả
df_results = pd.DataFrame([
    {
        'STT': i+1,
        'File Name': r['file_name'],
        'Feature Dim': r['features_shape'][0]
    } 
    for i, r in enumerate(results)
])

print("\n" + "="*50)
print("KẾT QUẢ XỬ LÝ 30 FILE ÂM THANH ĐẦU TIÊN")
print("="*50)
print(df_results.to_string(index=False))
print(f"\nTổng cộng: {len(results)} file được xử lý thành công")
print(f"Mỗi file được biểu diễn bằng vector {results[0]['features_shape'][0]} chiều")

[1/30] ✓ 001ca53d.wav - Shape: (58,)
[2/30] ✓ 00c9e799.wav - Shape: (58,)
[3/30] ✓ 015cf474.wav - Shape: (58,)
[4/30] ✓ 02ac1057.wav - Shape: (58,)
[5/30] ✓ 02f8e94d.wav - Shape: (58,)
[6/30] ✓ 034e4ffa.wav - Shape: (58,)
[7/30] ✓ 0395ba61.wav - Shape: (58,)
[8/30] ✓ 04076350.wav - Shape: (58,)
[9/30] ✓ 04490642.wav - Shape: (58,)
[10/30] ✓ 05a3154f.wav - Shape: (58,)
[11/30] ✓ 05d0dfa7.wav - Shape: (58,)
[12/30] ✓ 05d54a68.wav - Shape: (58,)
[13/30] ✓ 05e4028a.wav - Shape: (58,)
[14/30] ✓ 060e13d6.wav - Shape: (58,)
[15/30] ✓ 061cf689.wav - Shape: (58,)

✓ Thành công: 15/30 file

KẾT QUẢ XỬ LÝ 30 FILE ÂM THANH ĐẦU TIÊN
 STT    File Name  Feature Dim
   1 001ca53d.wav           58
   2 00c9e799.wav           58
   3 015cf474.wav           58
   4 02ac1057.wav           58
   5 02f8e94d.wav           58
   6 034e4ffa.wav           58
   7 0395ba61.wav           58
   8 04076350.wav           58
   9 04490642.wav           58
  10 05a3154f.wav           58
  11 05d0dfa7.wav           58


In [11]:
from qdrant_client.models import PointStruct

for i in range(len(results)):
    file_path = results[i]['file_path']
    features = results[i]['features']
    file_name = results[i]['file_name']
    label = results[i]['label']
    metadata = {
        "file_name": file_name,
        "file_path": file_path,
        "label": label
    }

    operation_info = client.upsert(
        collection_name="audio_features_v2",
        wait=True,
        points=[
            PointStruct(id = i, vector=features, payload= metadata ),
        ],
    )

    print(operation_info)

INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"


operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=4 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=5 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_fea

operation_id=6 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=7 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=8 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=9 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=10 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=11 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=12 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=13 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=14 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_fea

...

operation_id=471 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=472 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=473 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=474 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=475 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=476 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=477 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=478 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=479 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=480 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=481 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=482 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_fea

operation_id=483 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=484 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=485 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=486 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=487 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=488 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=489 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=490 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=491 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=492 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=493 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features_v2/points?wait=true "HTTP/1.1 200 OK"


operation_id=494 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=495 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=496 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=497 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=498 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=499 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=500 status=<UpdateStatus.COMPLETED: 'completed'>


In [ ]:
from pprint import pprint


test_path = r"C:\Users\ADZ\Downloads\wind_instruments_2s\00c9e799.wav"
test_features = audio_preprocessor.preprocess(test_path)
pprint(f"Test features shape: {test_features.shape}")
pprint(f"Test features: {test_features}")

search_result = client.query_points(
    collection_name="audio_features",
    query=test_features,
    with_payload=True,
    limit=3
).points



INFO:httpx:HTTP Request: POST http://localhost:6333/collections/audio_features/points/query "HTTP/1.1 200 OK"


'Test features shape: (58,)'
('Test features: [-1.53882072e-02 -8.47131602e-04 -3.21452422e-03 '
 '-3.75209795e-04\n'
 ' -7.92073871e-04  3.86296317e-04  6.38788549e-04 -7.29798289e-04\n'
 ' -1.20570588e-03 -5.11029188e-04  7.67558010e-04  1.31630900e-03\n'
 '  1.48698042e-03  9.07582586e-04 -3.77137350e-04 -1.41148982e-03\n'
 ' -5.90304894e-04  4.61677166e-04 -4.06007104e-05 -9.37975102e-04\n'
 '  1.17756630e-03  5.83972818e-04  3.15533225e-04  2.89316156e-04\n'
 '  3.82406017e-04  1.43642402e-04  1.51796948e-04  1.52728467e-04\n'
 '  2.27704149e-04  1.17228106e-04  1.83498300e-04  2.16768513e-04\n'
 '  2.65574597e-04  2.03341415e-04  1.54919184e-04  2.99509674e-04\n'
 '  1.49101769e-04  1.51793838e-04  2.16402786e-04  1.47450068e-04\n'
 '  8.48392086e-04  4.45863161e-04  1.61727637e-03  1.66908468e-03\n'
 '  1.40401895e-03  1.10917220e-03  1.87215308e-03  1.98971484e-04\n'
 '  1.53560134e-04  2.58710934e-04  2.50742136e-04  1.74517427e-04\n'
 '  1.45633032e-04  4.31722970e-04  2.3145

In [12]:

print(item for item in search_result)

<generator object <genexpr> at 0x00000297DC09F640>


In [19]:
for point in search_result:
    print(f"ID: {point.id}, Distance: {point.score}")
    print(f"Metadata: {point.payload}")

ID: 29, Distance: 0.92672205
Metadata: {'file_name': '106f028c.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\106f028c.wav'}
ID: 21, Distance: 0.9260608
Metadata: {'file_name': '0cea4550.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\0cea4550.wav'}
ID: 25, Distance: 0.91299057
Metadata: {'file_name': '0f958ff0.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\0f958ff0.wav'}


In [2]:
import os
import pandas as pd

# Load train.csv
df_train = pd.read_csv('train.csv')

# Get all .wav files from wind_instruments_1s folder
folder_path = r'C:\Users\ADZ\Downloads\wind_instruments_2s'
wind_files = set([f for f in os.listdir(folder_path) if f.endswith('.wav')])

# Filter train.csv to only include files in wind_instruments_1s
df_result = df_train[df_train['fname'].isin(wind_files)].reset_index(drop=True)

# Save to new CSV file
output_file = r'D:\Project\HCSDLDPT\wind_instruments_2s.csv'
df_result.to_csv(output_file, index=False)

print(f"Created {output_file} with {len(df_result)} files")
print(f"Total files in wind_instruments_2s folder: {len(wind_files)}")
print(f"Matched files from train.csv: {len(df_result)}")
print(df_result.head())

Created D:\Project\HCSDLDPT\wind_instruments_2s.csv with 700 files
Total files in wind_instruments_2s folder: 700
Matched files from train.csv: 700
          fname      label  manually_verified
0  001ca53d.wav  Saxophone                  1
1  00c9e799.wav       Oboe                  0
2  015cf474.wav   Clarinet                  0
3  02ac1057.wav      Flute                  0
4  02f8e94d.wav   Clarinet                  0


In [8]:
import pandas as pd
from qdrant_client.models import PointStruct

# Load the wind_instruments_1s.csv to create a mapping of filename to label
wind_csv_path = 'wind_instruments_1s.csv'
wind_df = pd.read_csv(wind_csv_path)

# Create a dictionary mapping filename to label for quick lookup
fname_to_label = dict(zip(wind_df['fname'], wind_df['label']))

# Update the upsert loop with label from CSV
for i in range(len(results)):
    file_path = results[i]['file_path']
    features = results[i]['features']
    file_name = results[i]['file_name']
    
    # Get label from wind_instruments_1s.csv if the file exists in it
    label = fname_to_label.get(file_name, "unknown")
    
    metadata = {
        "file_name": file_name,
        "file_path": file_path,
        "label": label,
    }

    operation_info = client.upsert(
        collection_name="audio_features",
        wait=True,
        points=[
            PointStruct(id=i, vector=features, payload=metadata),
        ],
    )

    print(operation_info)

INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "

operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=4 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=5 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=6 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=7 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=8 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=9 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=10 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=11 status=<UpdateStatus.COMPLETED: 'completed'>


INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT http://localhost:6333/collections/audio_features/points?wait=true "HTTP/1.1 200 OK"


operation_id=12 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=13 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=14 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=15 status=<UpdateStatus.COMPLETED: 'completed'>


In [9]:
test_path = r"D:\Project\HCSDLDPT\wind_instruments_1s\213d2998.wav"
test_features = audio_preprocessor.preprocess(test_path)
search_result = client.query_points(
    collection_name="audio_features",
    query=test_features,
    with_payload=True,
    limit=3
).points

# print(search_result) 

INFO:httpx:HTTP Request: POST http://localhost:6333/collections/audio_features/points/query "HTTP/1.1 200 OK"


In [10]:
for point in search_result:
    print(f"ID: {point.id}, Distance: {point.score}")
    print(f"Metadata: {point.payload}")

ID: 7, Distance: 0.9068119
Metadata: {'file_name': '064cfad3.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\064cfad3.wav', 'label': 'Clarinet'}
ID: 3, Distance: 0.9050892
Metadata: {'file_name': '05a3154f.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\05a3154f.wav', 'label': 'Oboe'}
ID: 13, Distance: 0.90431905
Metadata: {'file_name': '085652af.wav', 'file_path': 'D:\\Project\\HCSDLDPT\\wind_instruments_1s\\085652af.wav', 'label': 'Oboe'}
